# 02 — Calibrating TCI into a Probability

**Goal.** Make `TCI = 0.7` mean *"70% of frames that score this are genuine
threats"*. Today it means nothing in particular — it is a regression output
against a synthetic target.

A calibrated TCI is defensible in a way `R-squared = 0.984` on self-generated
labels is not: you can state its meaning in one sentence, plot a reliability
diagram, and report expected calibration error.

## Prerequisite

This notebook consumes `var/events/*.jsonl`, produced by running the pipeline
over labelled footage:

```bash
python tools/replay.py --clips data/clips --labels data/clips/labels.csv
```

If you have no event logs yet, the cells below fall back to a synthetic
demonstration set and say so loudly. **Do not report numbers from that path.**

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "sentrix").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import numpy as np
import pandas as pd

In [ ]:
from sentrix.pipeline.events import load_dataset
from sentrix.paths import EVENTLOG_DIR

df = load_dataset(EVENTLOG_DIR)
REAL = len(df) > 0 and "label" in df.columns and df["label"].notna().any()

if REAL:
    print(f"loaded {len(df):,} labelled frames from {df['log_file'].nunique()} log(s)")
else:
    print("!! NO LABELLED EVENT LOGS FOUND - using a synthetic demo set.")
    print("!! Numbers below are illustrative only. Do not put them in the report.")
    rng = np.random.default_rng(42)
    n = 4000
    y = rng.integers(0, 2, n)
    df = pd.DataFrame({
        "vision":    np.clip(rng.beta(np.where(y == 1, 5, 1.5), 2, n), 0, 1),
        "audio":     np.clip(rng.beta(np.where(y == 1, 3, 1.2), 4, n), 0, 1),
        "motion":    np.clip(rng.beta(np.where(y == 1, 4, 1.5), 3, n), 0, 1),
        "behaviour": np.clip(rng.beta(np.where(y == 1, 4, 1.2), 3, n), 0, 1),
        "identity":  np.clip(rng.beta(np.where(y == 1, 5, 1.2), 2, n), 0, 1),
        "weapon":    np.clip(rng.beta(np.where(y == 1, 2, 1.05), 8, n), 0, 1),
        "fire":      np.clip(rng.beta(1.05, 12, n), 0, 1),
        "is_night":  rng.integers(0, 2, n).astype(float),
        "label":     np.where(y == 1, "threat", "normal"),
    })

df.head()

## Split by source clip, never by frame

Adjacent frames from one clip are near-duplicates. Splitting at frame level leaks
the validation set into training and inflates every metric.

This is not hypothetical for this project: the V3 violence run split RWF-2000 at
frame level, and **17.7% of its validation source videos also appear in training**.
The same mistake here would invalidate the calibration.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sentrix.fusion.features import CONTRACT_V2

group_col = "log_file" if REAL and "log_file" in df.columns else None
if group_col is None:
    df["_group"] = np.arange(len(df)) // 50       # synthetic pseudo-clips
    group_col = "_group"

X = df[list(CONTRACT_V2)].astype(float).values
y = (df["label"].astype(str).str.lower()
     .isin({"threat", "true", "1", "fight", "weapon", "fire", "intrusion"})).astype(int).values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, valid_idx = next(splitter.split(X, y, groups=df[group_col]))
print(f"train {len(train_idx):,} frames | valid {len(valid_idx):,} frames")
print("groups disjoint:",
      set(df[group_col].iloc[train_idx]).isdisjoint(set(df[group_col].iloc[valid_idx])))
print(f"threat prevalence: train {y[train_idx].mean():.3f} | valid {y[valid_idx].mean():.3f}")

## Score the validation set with the current engine

We use the shipped fusion path as-is. The point is to calibrate whatever the system
currently outputs, not to replace it.

In [ ]:
from sentrix.fusion.tci import TCIEngine
from sentrix.fusion.features import FeatureVector

scores = []
for row in X[valid_idx]:
    fv = FeatureVector(**dict(zip(CONTRACT_V2, row)))
    fv.person_present = bool(fv.vision > 0.35)
    # Fresh engine per row: we want the instantaneous score, not the EMA.
    scores.append(TCIEngine().compute(fv, now=1.0).raw_tci)
scores = np.clip(np.array(scores), 0, 1)
y_valid = y[valid_idx]
print(f"scored {len(scores):,} frames | mean {scores.mean():.3f} "
      f"| range [{scores.min():.3f}, {scores.max():.3f}]")

In [ ]:
from sentrix.fusion.calibration import (IsotonicCalibrator, PlattCalibrator,
                                        expected_calibration_error)
from sklearn.model_selection import train_test_split

# Split the validation scores again: fit the calibrator on one half and measure
# ECE on the other. Fitting and evaluating on the same scores would report a
# calibration error of essentially zero.
fit_s, test_s, fit_y, test_y = train_test_split(
    scores, y_valid, test_size=0.5, random_state=0, stratify=y_valid)

results = {"uncalibrated": test_s}
try:
    results["platt"] = PlattCalibrator.fit(fit_s, fit_y).transform(test_s)
    results["isotonic"] = IsotonicCalibrator.fit(fit_s, fit_y).transform(test_s)
except Exception as exc:
    print("calibrator fitting failed:", exc)

pd.DataFrame([{"method": k, "ECE": round(expected_calibration_error(v, test_y), 4)}
              for k, v in results.items()]).sort_values("ECE")

## Reliability diagram

Perfect calibration is the diagonal. Above it, the model is under-confident;
below it, over-confident. Over-confidence is the dangerous direction for a
security system — it means TCI 0.9 events are mostly false alarms.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
edges = np.linspace(0, 1, 11)
centres = (edges[:-1] + edges[1:]) / 2
for name, probs in results.items():
    acc = [test_y[(probs > lo) & (probs <= hi)].mean()
           if ((probs > lo) & (probs <= hi)).any() else np.nan
           for lo, hi in zip(edges[:-1], edges[1:])]
    ax.plot(centres, acc, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax.set_xlabel("predicted TCI"); ax.set_ylabel("observed threat frequency")
ax.set_title("Reliability diagram" + ("" if REAL else "  [SYNTHETIC DEMO]"))
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## Saving

Writing `models/tci_calibrator.json` makes `TCIEngine` pick it up automatically on
next start — no code change.

The guard below refuses to save from the synthetic path. A calibrator fitted on
invented data would be worse than none, because it would look legitimate.

In [ ]:
from sentrix.fusion.calibration import IsotonicCalibrator, save
from sentrix.paths import MODELS_DIR

if not REAL:
    print("REFUSING to save: no real labelled event logs were loaded.")
else:
    calibrator = IsotonicCalibrator.fit(fit_s, fit_y)
    save(calibrator, MODELS_DIR / "tci_calibrator.json")
    print("saved. TCI is now a calibrated probability.")